<a href="https://colab.research.google.com/github/sergo8online/IOAI-Chicken-Counting-Problem-Solution/blob/main/Chicken_Counting_Solution_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_from_disk
import logging
from torchvision import transforms
from tqdm import tqdm
import numpy as np
import math

#Contestants should mount "counting_problem_train(v1)" datasets while creating the node.
TRAIN_PATH = "./"  #Address of the training set and base.pth
# The training set is deployed automatically in the testing machine.
# You notebook can access the TRAIN_PATH even if you do not mount it along with notebook.
TRAINING_SET = TRAIN_PATH + "train" #Address of the traninig set
BASE_MODEL_PATH = "/content/IOAI-2025/Individual-Contest/Chicken_Counting/base.pth"#Address of the .pth file
DTYPE = torch.float32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
scale = 300.0

In [ ]:
!git clone --depth 1 https://github.com/IOAI-official/IOAI-2025.git
import os
from pathlib import Path

base_path = Path("/content/IOAI-2025/Individual-Contest/Radar/training_set")

if base_path.exists():
    files = list(base_path.glob("*.mat.pt"))
    print(f"Yes: {len(files)}")
else:
    print("Nooo.")

In [ ]:
def logging_level(level='info'):
    str_format = '%(asctime)s - %(levelname)s: %(message)s'
    if level == 'debug':
        logging.basicConfig(level=logging.DEBUG, format=str_format, datefmt='%Y-%m-%d %H:%M:%S')
    elif level == 'info':
        logging.basicConfig(level=logging.INFO, format=str_format, datefmt='%Y-%m-%d %H:%M:%S')
    return logging


class BatchLossLogger:
    def __init__(self, log_interval=100):
        self.losses = []
        self.batch_number = 0
        self.log_interval = log_interval

    def log(self, loss):
        self.losses.append(loss)
        self.batch_number += 1
        if self.batch_number % self.log_interval == 0:
            logging.info(f'Batch No. {self.batch_number:7d} - loss: {loss:.6f}')

In [ ]:
!pip install crowdcount
from crowdcount.models import UNet

print(dir(UNet.modules))


class FeatureExtraction(nn.Module):
    def __init__(self, in_channels=3):
        super(FeatureExtraction, self).__init__()   #(3, 720, 1280)
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=2, dilation=2) #(64, 720, 1280)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=2, dilation=2)          #(64, 720, 1280)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)                 #(64, 360, 640)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=2, dilation=2)         #(128, 360, 640)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=2, dilation=2)        #(128, 360, 640)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)                 #(128, 180, 320)


    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool4(x)

        return x   #shape is (batch_size, 128, 180, 320)

    def load_pretrained_weights_partial(self, weights_path, num_layers=4):
        save_model = torch.load(weights_path)
        partial_state_dict = {}
        expected_layers = [
            'feature_extraction.conv1.weight', 'feature_extraction.conv1.bias',
            'feature_extraction.conv2.weight', 'feature_extraction.conv2.bias',
            'feature_extraction.conv3.weight', 'feature_extraction.conv3.bias',
            'feature_extraction.conv4.weight', 'feature_extraction.conv4.bias',
        ]
        state_dict = {k.split('.', 1)[-1]: v for k, v in save_model.items() if k in expected_layers[:2 * num_layers]}
        model_dict = self.state_dict()

        for k in state_dict:
            if k in model_dict:
                partial_state_dict[k] = state_dict[k]
                print(k)

        self.load_state_dict(partial_state_dict)

class inconv(nn.Module):
  def __init__(self, in_c, out_c):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(in_c, out_c, 3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.Conv2d(out_c, out_c, 3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU()
    )

  def forward(self, x):
    return self.conv(x)

class UNetV1(UNet):
  def __init__(self, in_channels=3, out_channels=1):
    super().__init__()
    self.inc = inconv(in_channels, 64)
    self.outc = nn.Conv2d(64, out_channels, kernel_size=1)


class ChickenCounting(nn.Module):
    def __init__(self):
        super(ChickenCounting, self).__init__()
        self.feature_extraction = FeatureExtraction()
        # self.feature_decoder = DensityDecoder()
        self.unet = UNetV1(in_channels=128, out_channels=1)

    def forward(self, x):
        x = self.feature_extraction(x)
        # x = self.feature_decoder(x)
        x = self.unet(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureExtraction(nn.Module):
    def __init__(self, in_channels=3):
        super(FeatureExtraction, self).__init__()   #(3, 720, 1280)
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=2, dilation=2) #(64, 720, 1280)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=2, dilation=2)          #(64, 720, 1280)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)                 #(64, 360, 640)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=2, dilation=2)         #(128, 360, 640)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=2, dilation=2)        #(128, 360, 640)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)                 #(128, 180, 320)


    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool4(x)

        return x   #shape is (batch_size, 128, 180, 320)

    def load_pretrained_weights_partial(self, weights_path, num_layers=4):
        save_model = torch.load(weights_path)
        partial_state_dict = {}
        expected_layers = [
            'feature_extraction.conv1.weight', 'feature_extraction.conv1.bias',
            'feature_extraction.conv2.weight', 'feature_extraction.conv2.bias',
            'feature_extraction.conv3.weight', 'feature_extraction.conv3.bias',
            'feature_extraction.conv4.weight', 'feature_extraction.conv4.bias',
        ]
        state_dict = {k.split('.', 1)[-1]: v for k, v in save_model.items() if k in expected_layers[:2 * num_layers]}
        model_dict = self.state_dict()

        for k in state_dict:
            if k in model_dict:
                partial_state_dict[k] = state_dict[k]
                print(k)

        self.load_state_dict(partial_state_dict)

class DoubleConv(nn.Module):
  def __init__(self, in_c, out_c):
    super().__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(in_c, out_c, 3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.Conv2d(out_c, out_c, 3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU()
    )

  def forward(self, x):
    return self.conv(x)

class Inconve(nn.Module):
  def __init__(self, in_c, out_c):
    super().__init__()
    self.inconve = DoubleConv(in_c, out_c)
  def forward(self, x):
    return self.inconve(x)

class Down(nn.Module):
  def __init__(self, in_c, out_c):
    super().__init__()
    self.down = nn.Sequential(
        nn.MaxPool2d(2),
        DoubleConv(in_c, out_c)
    )
  def forward(self, x):
    return self.down(x)

class Up(nn.Module):
  def __init__(self, in_c, out_c, bilinear=False):
    super().__init__()
    if bilinear:
      self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    else:
      self.up = nn.ConvTranspose2d(in_c//2, in_c//2, 2, stride=2)
    self.conv = DoubleConv(in_c, out_c)

  def forward(self, x1, x2):
    x1 = self.up(x1)
    diffY = x2.size()[2] - x1.size()[2]
    diffX = x2.size()[3] - x1.size()[3]
    x1 = F.pad(x1, (diffX//2, diffX-(diffX//2), diffY//2, diffY-(diffY//2)))

    x = torch.cat([x1, x2], dim=1)
    return self.conv(x)

class Out(nn.Module):
  def __init__(self, in_c, out_c):
    super().__init__()
    self.conv = nn.Conv2d(in_c, out_c, 1)
  def forward(self, x):
    return self.conv(x)


class MyUnet(nn.Module):
  def __init__(self, in_c, out_c, bilinear=False):
    super().__init__()
    self.init = Inconve(in_c, 64)
    self.down1 = Down(64, 128)
    self.down2 = Down(128, 256)
    self.down3 = Down(256, 512)
    self.down4 = Down(512, 512)
    self.up1 = Up(1024, 256)
    self.up2 = Up(512, 128)
    self.up3 = Up(256, 64)
    self.up4 = Up(128, 64)
    self.out = Out(64, out_c)

  def forward(self, x):
    x1 = self.init(x)
    x2 = self.down1(x1)
    x3 = self.down2(x2)
    x4 = self.down3(x3)
    x5 = self.down4(x4)
    x = self.up1(x5, x4)
    x = self.up2(x, x3)
    x = self.up3(x, x2)
    x = self.up4(x, x1)
    x = self.out(x)
    return x

class ChickenCounting(nn.Module):
    def __init__(self):
        super(ChickenCounting, self).__init__()
        self.feature_extraction = FeatureExtraction()
        # self.feature_decoder = DensityDecoder()
        self.unet = MyUnet(128, 1)

    def forward(self, x):
        x = self.feature_extraction(x)
        # x = self.feature_decoder(x)
        x = self.unet(x)
        return x

In [ ]:
from datasets import load_dataset

# 从 Hugging Face 加载数据集
train_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Train2",
                            data_dir="train",
                            split="train")

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.452016860247, 0.447249650955, 0.431981861591],
                        std=[0.23242045939, 0.224925786257, 0.221840232611])
])

def collate_fn(batch, scale=scale):
    return {
        "image": torch.stack([image_transform(item["image"]) for item in batch]),
        "density": torch.stack([torch.tensor(item["density"], dtype=DTYPE).unsqueeze(0) * scale for item in batch])
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

In [ ]:
#Definition of the training process
def train_chickenfcn(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device, save_path):
    model.train()
    criterion_mse = torch.nn.MSELoss(reduction='sum').to(device)
    criterion_mae = torch.nn.L1Loss(reduction='sum').to(device)
    best_loss = float('inf')
    print(train_loader.__len__())

    for epoch in range(num_epochs):
        train_loss_mse = 0.0
        train_loss_mae = 0.0

        train_loader_tqdm = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=False)

        for i, data in enumerate(train_loader_tqdm, 0):
            inputs, targets = data["image"], data["density"]
            inputs = inputs.to(device).float()
            targets = targets.to(device).float()
            # print(targets.shape)
            # t = np.sum((targets[0] / scale).cpu().numpy().squeeze())
            # print(t)

            optimizer.zero_grad()

            # z2, z3, z4, out = model(inputs)
            # masks_z2 = F.interpolate(targets, size=z2.shape[2:], mode='nearest')
            # masks_z3 = F.interpolate(targets, size=z3.shape[2:], mode='nearest')
            # masks_z4 = F.interpolate(targets, size=z4.shape[2:], mode='nearest')

            # loss_z2 = loss_fn(z2, masks_z2)
            # loss_z3 = loss_fn(z3, masks_z3)
            # loss_z4 = loss_fn(z4, masks_z4)
            # loss_final = loss_fn(out, targets)

            # loss = loss_z2 + loss_z3 + loss_z4 + loss_final
            # loss.backward()

            outputs = model(inputs)
            loss_mse = criterion_mse(outputs, targets)
            loss_mae = criterion_mae(outputs, targets)
            loss_mse.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)

            optimizer.step()
            train_loss_mse += loss_mse.item()
            train_loss_mae += loss_mae.item()

            train_loader_tqdm.set_postfix({'Train MSE Loss': loss_mse.item(), 'Train MAE Loss': loss_mae.item()})

        train_loss_mse /= (len(train_loader))
        train_loss_mae /= (len(train_loader))
        logging.info(
            f'Epoch [{epoch + 1}/{num_epochs}], Train MSE loss: {train_loss_mse:.8f}, MAE loss: {train_loss_mae:.8f}')

        scheduler.step()

        # Validation
        model.eval()
        val_loss_mse = 0.0
        val_loss_mae = 0.0

        val_loader_tqdm = tqdm(val_loader, desc=f'Validation Epoch {epoch + 1}/{num_epochs}', leave=False)

        with torch.no_grad():
            for i, data in enumerate(val_loader_tqdm, 0):
                inputs, targets = data["image"], data["density"]
                inputs = inputs.to(device).float()
                targets = targets.to(device).float()

                # z2, z3, z4, out = model(inputs)
                # masks_z2 = F.interpolate(targets, size=z2.shape[2:], mode='nearest')
                # masks_z3 = F.interpolate(targets, size=z2.shape[2:], mode='nearest')
                # masks_z4 = F.interpolate(targets, size=z2.shape[2:], mode='nearest')

                # loss_z2 = loss_fn(z2, masks_z2)
                # loss_z3 = loss_fn(z3, masks_z3)
                # loss_z4 = loss_fn(z4, masks_z4)
                # loss_final = loss_fn(out, targets)
                # loss = loss_z2 + loss_z3 + loss_z4 + loss_final

                # val_loss+=loss.item()

                outputs = model(inputs)
                mse_loss = criterion_mse(outputs, targets)
                mae_loss = criterion_mae(outputs, targets)
                val_loss_mse += mse_loss.item()
                val_loss_mae += mae_loss.item()

                val_loader_tqdm.set_postfix(
                    {'Validation MSE Loss': mse_loss.item(), 'Validation MAE Loss': mae_loss.item()})


        val_loss_mse /= (len(val_loader))
        val_loss_mae /= (len(val_loader))
        logging.info(
            f'Epoch [{epoch + 1}/{num_epochs}], Validation MSE Loss: {val_loss_mse:.8f}, MAE Loss: {val_loss_mae:.8f}')


        #Save Model
        if val_loss_mae < best_loss:
            best_loss = val_loss_mae
            torch.save(model.state_dict(), save_path)


    # print('Finished Training ChickenFCN')
    print("Finished Training TEDNet")

In [ ]:
logging = logging_level('info')
logging.debug('use debug level logging setting')

################################################################################
# Experiment Settings
################################################################################
learning_rate = 1e-4
lr_decay = 1e-5
weight_decay = 0.0001
save_path = "model.pth"

epochs = 30

# Training
# model = ChickenCounting().to(DEVICE)
model = ChickenCounting().to(DEVICE)
model.feature_extraction.load_pretrained_weights_partial(BASE_MODEL_PATH)
# for param in model.feature_extraction.parameters():
#   param.requires_grad=False
# model.feature_extraction.eval()
print('load model success')

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=1 - lr_decay)

logging.info('Begin training single view model...')
train_chickenfcn(model, train_loader, val_loader, optimizer, scheduler, epochs, DEVICE, save_path=save_path)
logging.info('Finished training single view model.')

In [ ]:
# Definition of the evaluation function
def evaluate(model, val_loader, device, scale): # Function used for final scoring.
    model.eval()  # Set the model to evaluation mode

    # Initialize metrics
    mse = 0.0
    mae = 0.0
    predict_num = 0.0
    true_num = 0.0
    rate = 0.0

    with torch.no_grad():  # Disable gradient calculation for inference
        for i, data in enumerate(val_loader, 0):
            inputs, targets = data["image"], data["density"]
            inputs = inputs.to(device).float()  # Move inputs to device and convert to float
            targets = targets.to(device).float()  # Move targets to device and convert to float

            # Get the model predictions
            outputs = model(inputs) / scale  # Adjusting for the scaling factor

            # Convert tensors to numpy for visualization and metrics calculation
            inputs_np = inputs.cpu().numpy()  # Convert inputs to numpy
            targets_np = targets.cpu().numpy()  # Convert targets to numpy
            outputs_np = outputs.cpu().numpy()  # Convert outputs to numpy
            # imshow_res(inputs_np, targets_np, outputs_np, scale)  # Uncomment to visualize results

            # Calculate true and predicted sums for comparison
            t = np.sum((targets[0] / scale).cpu().numpy().squeeze())  # Ground truth sum
            g = np.sum(outputs.cpu().numpy().squeeze())*scale  # Predicted sum
            print(f'NO.{i}   true_sum={t}, get_sum={g}, abs={abs(t - g)}, rate={abs(1 - g / t)}')

            # Update metrics
            predict_num += g
            true_num += t
            rate += abs(1 - g / t)
            mae += abs(t - g)
            mse += abs(t - g) * abs(t - g)

    # Calculate average metrics across all batches
    mae /= len(val_loader)
    mse /= len(val_loader)
    predict_num /= len(val_loader)
    true_num /= len(val_loader)
    rate /= len(val_loader)

    # Log the results
    logging.info(
        f'test ---- Score: {math.exp(-rate):.3f}, MSE: {mse:.4f}, MAE: {mae:.4f}, Chicken_avg: {predict_num:.4f}')
    return math.exp(-rate)

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=DEVICE))
model.to(DEVICE)
evaluate(model, val_loader, DEVICE, scale)

In [ ]:
from datasets import load_dataset

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test",
                            data_dir="valandtest",
                            split="validation")

def collate_fn(batch): # The test datasets will not provide target densities
    return torch.stack([image_transform(item["image"]) for item in batch])

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / scale
        outputs = torch.relu(outputs)
        predictions.append(outputs.cpu().numpy())

pred_a = np.concatenate(predictions, axis=0)

del test_dataset
del test_loader
del predictions

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test",
                            data_dir="valandtest",
                            split="test")
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / scale
        outputs = torch.relu(outputs)
        predictions.append(outputs.cpu().numpy())

pred_b = np.concatenate(predictions, axis=0)

np.savez('submission.npz', pred_a=pred_a, pred_b=pred_b) # save your submissions in `submission.npz` file with the keys `pred_a` and `pred_b`

In [ ]:
%run /content/IOAI-2025/Individual-Contest/Chicken_Counting/metrics.py